In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [3]:
from dotenv import load_dotenv
import os 
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
            

In [4]:
documents= load_faq_data()
index = build_index(documents)


rag_openai = RAGBase(
    index=index,
    llm_client=openai_client,
    model='gpt-5.4-mini',
    api_type='openai'
)

rag_groq = RAGBase(
    index=index,
    llm_client=groq_client,
    model='qwen/qwen3.6-27b',
    api_type='groq'
)

In [5]:
question = 'I just discovered the course, can i still join?'

openai_answer = rag_openai.rag(question)
qwen_answer = rag_groq.rag(question)

print("=== OpenAI / GPT-5.4-mini ===")
print(openai_answer)

print("\n=== Groq / Qwen 3.6 27B ===")
print(qwen_answer)

=== OpenAI / GPT-5.4-mini ===
Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.

=== Groq / Qwen 3.6 27B ===

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "I just discovered the course, can i still join?"
   - **Context Provided:** A list of Q&A pairs under "General Course-Related Questions" and "Module 3: Orchestration".
   - **Task:** Answer the question based *only* on the provided context. If not found, respond with "I don't know."

2.  **Scan Context for Keywords:**
   - Keywords: "just discovered", "course", "join"
   - Found exact match in the first Q&A pair:
     - Q: I just discovered the course. Can I still join?
     - A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

3.  **Formulate Answer:**
   - Extract the exact answer from the context.
   - Ensure it directly addresses the question.
